# GRU-Based Anomaly Detection on Synthetic Telemetry Data

## Research Methods Project Update

This notebook implements an unsupervised anomaly detection pipeline using a **GRU autoencoder** on synthetically generated telemetry data.

The current scope is limited to **anomaly detection on synthetic telemetry data only**.  
The **data ingestion pipeline is not included yet**.

## Objective

The goal of this notebook is to detect unusual sequential patterns in synthetic telemetry data using a **Gated Recurrent Unit (GRU)** based autoencoder.

Because telemetry events are time-ordered and contain temporal dependencies, GRU is suitable for learning sequence behavior across records. Since no confirmed anomaly labels are used for training, this task is treated as **unsupervised anomaly detection**.

### What this notebook does
1. Loads and sorts the synthetic telemetry data by time  
2. Selects appropriate numerical features for modeling  
3. Scales the data and builds fixed-length sequences  
4. Trains a GRU autoencoder to learn normal sequence patterns  
5. Uses reconstruction error to assign anomaly scores  
6. Flags high-error sequences as anomalies  
7. Saves the final results to a CSV file

## Dataset Description

The dataset used in this notebook is a **synthetic telemetry dataset** generated for anomaly detection experiments.

### Main types of information in the dataset
- source type
- device
- event type
- severity
- time-based features
- encoded identifiers
- rolling behavioral indicators
- value-related features

### Important note
This dataset already contains several engineered numerical features, which makes it more suitable for direct sequence modeling than the earlier raw telemetry dataset.

## Why GRU for This Task?

A **GRU (Gated Recurrent Unit)** is a type of recurrent neural network designed for sequence data.

GRUs are useful here because telemetry data is naturally **time-ordered**. Instead of treating each record independently, the model learns patterns across a sequence of events.

### Why GRU was chosen
- it captures **temporal dependencies** in sequential data
- it works well for **time-series and event-sequence modeling**
- it is simpler and lighter than LSTM while still modeling sequence behavior well

In this notebook, GRU is used inside an **autoencoder** architecture to learn normal behavior and identify unusual sequence patterns.

## Why an Autoencoder for Anomaly Detection?

This project uses a **GRU autoencoder** because the dataset does not rely on explicit anomaly labels for supervised training.

### Core idea
The model is trained to reconstruct input sequences that are assumed to be mostly normal.  
If the model later sees a sequence that is unusual, it will generally reconstruct it poorly.

That reconstruction gap becomes the **anomaly score**.

### Reconstruction error
After training:
- **Low reconstruction error** → sequence looks normal
- **High reconstruction error** → sequence may be anomalous

This makes the method suitable for **unsupervised anomaly detection**.

## Step 1: Import Libraries

This section imports the required Python libraries for:
- data handling with **pandas** and **numpy**
- feature scaling with **scikit-learn**
- building the GRU autoencoder with **TensorFlow / Keras**
- training control with **EarlyStopping**

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

## Step 2: Set Random Seeds

This step sets the random seeds for NumPy and TensorFlow so that the experiment is more reproducible.

Using fixed seeds helps reduce variation across repeated runs of the notebook.

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

## Step 3: Load the Synthetic Telemetry Data

The dataset used in this notebook is a synthetic telemetry dataset generated for anomaly detection experiments.

### Main types of information in the dataset
- source type
- device
- event type
- severity
- time-based features
- encoded identifiers
- rolling behavioral indicators
- value-related features

This dataset is already more structured than the earlier raw telemetry dataset and includes engineered numerical features that are suitable for sequence-based modeling.

In [4]:
DATA_PATH = "ctgan_mixed_training_data.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (19747, 31)
Columns: ['source_type', 'device', 'event_type', 'severity', 'ts_unix', 'hour_of_day', 'day_of_week', 'minute_of_day', 'time_delta_sec', 'event_id', 'device_id', 'severity_id', 'source_id', 'prev_was_error', 'rolling_error_rate', 'login_fail_streak', 'login_fail_prev5', 'cpu_mem_prev5', 'config_change_prev5', 'prior_config_change', 'prior_login_failure', 'prior_cpu_high', 'prior_memory_high', 'prior_service_error', 'prior_interface_down', 'prior_kernel_panic', 'value_observed', 'value_filled', 'value_zscore_local', 'value_local_anomaly', 'timestamp_unix']


## Step 4: Sort the Data by Time

The dataset is sorted using `timestamp_unix` so that all events are in chronological order.

This is important because GRU models learn from **ordered sequences**.  
If the events are not in the correct time order, the sequence relationships would be incorrect.

In [5]:
df = df.sort_values("timestamp_unix").reset_index(drop=True)

## Step 5: Select Model Input Features

The dataset contains many columns, but not all of them should be used as model inputs.

### What is included
The selected features focus on:
- time-based variables
- encoded identifiers
- rolling behavioral features
- prior event indicators
- value-related features

### What is excluded
Some columns are intentionally excluded, especially:
- raw categorical text columns
- duplicate representations of the same information
- precomputed anomaly-like columns such as `value_local_anomaly`

### Why exclude `value_local_anomaly`?
Because it appears to contain anomaly-related information already. Including it would leak anomaly signals directly into the model and make the experiment less meaningful.

In [8]:
feature_cols = [
    "hour_of_day",
    "day_of_week",
    "minute_of_day",
    "time_delta_sec",
    "event_id",
    "device_id",
    "severity_id",
    "source_id",
    "prev_was_error",
    "rolling_error_rate",
    "login_fail_streak",
    "login_fail_prev5",
    "cpu_mem_prev5",
    "config_change_prev5",
    "prior_config_change",
    "prior_login_failure",
    "prior_cpu_high",
    "prior_memory_high",
    "prior_service_error",
    "prior_interface_down",
    "prior_kernel_panic",
    "value_observed",
    "value_filled",
    "value_zscore_local",
]

## Step 6: Prepare the Feature Matrix

After selecting the model input columns:
- missing values are filled where necessary
- the final feature matrix is converted to numeric format

This step ensures that the GRU model receives a clean numerical input matrix.

In [9]:
X = df[feature_cols].copy()
X = X.fillna(0).astype("float32")

print("Feature matrix shape:", X.shape)

Feature matrix shape: (19747, 24)


## Step 7: Feature Scaling and Training Split

Before training the model, the selected numerical features are standardized using **StandardScaler**.

### Why scaling is important
Different features can have very different numeric ranges.  
Scaling helps the GRU train more effectively by putting features on a more comparable scale.

### Training assumption
The first **60%** of the time-ordered data is used as the training portion under the assumption that it is **mostly normal**.

This is a practical assumption because the current setup is unsupervised and does not rely on confirmed anomaly labels.

In [10]:
split = int(0.6 * len(X))
X_train_raw = X.iloc[:split].values
X_all_raw = X.values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_all = scaler.transform(X_all_raw)

## Step 8: Build Sequences for GRU

GRU models learn from **sequences**, not isolated rows.

### Sequence settings used
- **Sequence length = 32**
- **Stride = 1**

Each input sample contains 32 consecutive telemetry records.

### Why sequences are needed
The model is learning temporal behavior across multiple steps.  
This is useful for telemetry anomaly detection because unusual behavior may only become visible across a short sequence of events rather than from a single record alone.

In [14]:
SEQ_LEN = 32
STRIDE = 1

def make_sequences(arr, seq_len=32, stride=1):
    seqs = []
    for i in range(0, len(arr) - seq_len + 1, stride):
        seqs.append(arr[i:i + seq_len])
    return np.array(seqs, dtype=np.float32)

X_train_seq = make_sequences(X_train, SEQ_LEN, STRIDE)
X_all_seq = make_sequences(X_all, SEQ_LEN, STRIDE)

print("Train sequences:", X_train_seq.shape)
print("All sequences:", X_all_seq.shape)

Train sequences: (11817, 32, 24)
All sequences: (19716, 32, 24)


## Step 9: GRU Autoencoder Architecture

The model used in this notebook is a **GRU autoencoder**.

### Why GRU?
A **GRU (Gated Recurrent Unit)** is a type of recurrent neural network designed for sequence data. It is well suited for telemetry records because the data is naturally **time-ordered**.

### Why an autoencoder?
The model is trained to reconstruct patterns that are assumed to be mostly normal.  
Sequences that are difficult to reconstruct are treated as potentially anomalous.

### Model structure
- **Encoder**
  - GRU layer with 64 units
  - GRU layer with 32 units
- **Bottleneck**
  - compressed sequence representation
- **Decoder**
  - RepeatVector
  - GRU layer with 32 units
  - GRU layer with 64 units
  - TimeDistributed Dense layer for final reconstruction

In [16]:
n_features = X_train_seq.shape[-1]

inputs = layers.Input(shape=(SEQ_LEN, n_features))

# Encoder
x = layers.GRU(64, return_sequences=True)(inputs)
x = layers.GRU(32, return_sequences=False)(x)

# Bottleneck -> Decoder
x = layers.RepeatVector(SEQ_LEN)(x)
x = layers.GRU(32, return_sequences=True)(x)
x = layers.GRU(64, return_sequences=True)(x)

outputs = layers.TimeDistributed(layers.Dense(n_features))(x)

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32, 64)         │        17,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 32, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 32, 32)         │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 32, 64)         │        18,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 32, 24)         │         1,560 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,400 (208.59 KB)

 Trainable params: 53,400 (208.59 KB)

 Non-trainable params: 0 (0.00 B)

## Step 10: Configure Early Stopping

An EarlyStopping callback is used to reduce overtraining.

If the validation loss stops improving, training is stopped early and the best model weights are restored.

In [17]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

## Step 11: Train the Model

The GRU autoencoder is trained by using the input sequence itself as the target output.

### Why?
Because this is an **autoencoder**, the goal is reconstruction:
- input = original sequence
- output = reconstructed version of the same sequence

### What to watch during training
Both **training loss** and **validation loss** should generally decrease over epochs.  
That indicates the model is learning useful sequence structure from the synthetic telemetry data.

In [18]:
history = model.fit(
    X_train_seq, X_train_seq,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 30s 231ms/step - loss: 0.9588 - val_loss: 1.0376
Epoch 2/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 18s 214ms/step - loss: 0.9371 - val_loss: 1.0192
Epoch 3/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 18s 190ms/step - loss: 0.9275 - val_loss: 1.0124
Epoch 4/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 18s 214ms/step - loss: 0.9235 - val_loss: 1.0081
Epoch 5/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 22s 258ms/step - loss: 0.9184 - val_loss: 1.0025
Epoch 6/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 34s 180ms/step - loss: 0.9143 - val_loss: 0.9967
Epoch 7/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 23s 202ms/step - loss: 0.9097 - val_loss: 0.9914
Epoch 8/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 20s 240ms/step - loss: 0.9053 - val_loss: 0.9889
Epoch 9/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 162ms/step - loss: 0.9028 - val_loss: 0.9895
Epoch 10/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 23s 190ms/step - loss: 0.9010 - val_loss: 0.9858
Epoch 11/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 18s 221ms/step - loss: 0.8984 - val_loss: 0.9848
Epoch 12/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 18

## Step 12: Compute Reconstruction Error

After training, the model reconstructs each sequence and the reconstruction error is calculated.

### Anomaly score
The anomaly score is defined as the **mean squared reconstruction error** across:
- all time steps
- all features in the sequence

### Interpretation
- lower score → sequence is more similar to learned normal behavior
- higher score → sequence is more unusual

In [21]:
def reconstruction_error(model, Xseq):
    pred = model.predict(Xseq, verbose=0)
    err = np.mean((Xseq - pred) ** 2, axis=(1, 2))
    return err

train_err = reconstruction_error(model, X_train_seq)
all_err = reconstruction_error(model, X_all_seq)

print("train_err shape:", train_err.shape)
print("all_err shape:", all_err.shape)
print("First 5 training errors:", train_err[:5])

train_err shape: (11817,)
all_err shape: (19716,)
First 5 training errors: [0.7215409  0.71881366 0.7285386  0.7205569  0.73634595]


## Step 13: Define the Anomaly Threshold

To convert anomaly scores into anomaly flags, a threshold is needed.

In this notebook, the threshold is set as the **99th percentile** of the training reconstruction error.

### Why this threshold?
The training set is assumed to be mostly normal, so very high reconstruction errors relative to that distribution are treated as anomalous.

### Interpretation
Any sequence with reconstruction error greater than this threshold is flagged as an anomaly.

In [22]:
THRESH_PCT = 99
threshold = np.percentile(train_err, THRESH_PCT)
seq_anom = (all_err > threshold).astype(int)

print(f"Threshold (p{THRESH_PCT}): {threshold:.6f}")
print("Anomalous sequences:", int(seq_anom.sum()), "out of", len(seq_anom))

Threshold (p99): 1.340345
Anomalous sequences: 733 out of 19716


## Step 14: Map Sequence-Level Results Back to Rows

The model detects anomalies at the **sequence level**, but for easier inspection, the results are mapped back to individual rows.

In this implementation:
- each sequence is associated with its **ending row**
- the anomaly score is attached to that ending row
- flagged anomalous sequences mark the ending row as anomalous

### Important note
This means the anomaly flag represents the endpoint of an anomalous sequence, not necessarily every row within that sequence.

In [25]:
row_flag = np.zeros(len(df), dtype=int)
row_score = np.zeros(len(df), dtype=float)

for i, (flag, score) in enumerate(zip(seq_anom, all_err)):
    end_idx = i + SEQ_LEN - 1
    row_score[end_idx] = max(row_score[end_idx], score)
    if flag == 1:
        row_flag[end_idx] = 1

df_out = df.copy()
df_out["gru_anomaly_score"] = row_score
df_out["gru_anomaly_flag"] = row_flag

print(df_out[["gru_anomaly_score", "gru_anomaly_flag"]].head())

   gru_anomaly_score  gru_anomaly_flag
0                0.0                 0
1                0.0                 0
2                0.0                 0
3                0.0                 0
4                0.0                 0


## Step 15: Save Final Results

The final output is saved as a CSV file.

This output includes the original telemetry records along with:
- **gru_anomaly_score**: reconstruction-based anomaly score
- **gru_anomaly_flag**: binary anomaly indicator

In [26]:
OUTPUT_FILE = "gru_ctgan_anomaly_results.csv"
df_out.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)

Saved: gru_ctgan_anomaly_results.csv


## Step 16: Inspect Detected Anomalies

After the anomaly scores and flags are generated, a quick summary is produced to inspect the detected anomalies.

This includes counts by:
- severity
- event type
- device

This step helps interpret what types of telemetry records are being flagged by the model.

In [27]:
anom = df_out[df_out["gru_anomaly_flag"] == 1]

print("\nTotal anomalous rows:", len(anom))
...


Total anomalous rows: 733


Ellipsis

## Results Summary

After running the notebook, the following key outputs are produced:

- Total number of rows in the dataset
- Number of model input features
- Number of training sequences
- Number of total sequences
- Reconstruction error threshold
- Number of detected anomalous sequences
- Output CSV file containing anomaly scores and flags

### Interpretation of current results
If the GRU model trains successfully and detects a small subset of unusual sequences, that indicates the anomaly detection pipeline is functioning as expected on the synthetic telemetry data.

## Limitations

This notebook provides a working anomaly detection pipeline, but there are important limitations:

1. **Synthetic data only**
   - the dataset is not real production telemetry

2. **No ground-truth anomaly labels**
   - flagged anomalies are model-based candidates rather than confirmed anomalies

3. **Training data assumption**
   - the first 60% of the data is assumed to be mostly normal

4. **Threshold is heuristic**
   - the 99th percentile is a practical choice, but not the only possible threshold

5. **Sequence-end mapping**
   - row-level anomaly flags correspond to sequence endpoints

6. **No ingestion pipeline yet**
   - this notebook focuses only on anomaly detection on synthetic telemetry data

## Conclusion

This notebook demonstrates a complete **GRU-based anomaly detection workflow** on synthetically generated telemetry data.

The data was sorted, selected into model-relevant numeric features, standardized, converted into sequential inputs, and used to train a GRU autoencoder in an unsupervised setting. Reconstruction error was then used to identify unusual sequences.

### Final takeaway
The anomaly detection pipeline is working on synthetic telemetry data and is ready to be discussed as a project update.

### Future work
- inspect flagged anomalies more carefully
- compare results with other sequence-based models
- explore richer telemetry datasets
- later integrate the data ingestion pipeline

## Presentation Note

This notebook is intended to serve as both:
- the implementation
- the documentation for the current project update

No separate presentation or report is required for this stage unless requested later.

## Final Output Snapshot

The cell below prints the key summary values from the experiment so they can be referenced easily during discussion with the professor.

In [28]:
print("=== Final Experiment Summary ===")
print("Total rows:", len(df))
print("Total features:", len(feature_cols))
print("Train sequences:", X_train_seq.shape)
print("All sequences:", X_all_seq.shape)
print(f"Threshold (p{THRESH_PCT}): {threshold:.6f}")
print("Anomalous sequences detected:", int(seq_anom.sum()))
print("Results saved to:", OUTPUT_FILE)

=== Final Experiment Summary ===
Total rows: 19747
Total features: 24
Train sequences: (11817, 32, 24)
All sequences: (19716, 32, 24)
Threshold (p99): 1.340345
Anomalous sequences detected: 733
Results saved to: gru_ctgan_anomaly_results.csv
